# 02 · Descriptors: The Magic Behind `property`, Methods & ORMs

**Module 10 – Advanced Python: Internals & Metaprogramming** · Estimated time: 50 min

### 🎯 What you will learn
- The descriptor protocol: `__get__`, `__set__`, `__delete__`, `__set_name__`
- Data vs non-data descriptors (and why it matters for lookup, lesson 01)
- Building **reusable validated fields** (how Django models, SQLAlchemy and Pydantic-like libraries work)
- Re-implementing `property`, `classmethod`, `staticmethod` and `cached_property` yourself
- Where per-instance data should be stored
- `__slots__` are descriptors too

## 1. The protocol

A **descriptor** is any object that defines at least one of these methods and is stored as a **class attribute**:

| Method | Called for | Example |
|---|---|---|
| `__get__(self, instance, owner)` | `obj.attr` / `Class.attr` | read |
| `__set__(self, instance, value)` | `obj.attr = value` | write |
| `__delete__(self, instance)` | `del obj.attr` | delete |
| `__set_name__(self, owner, name)` | once, when the class is created | learns its attribute name |

- **Data descriptor**: defines `__set__` or `__delete__` → **wins** over the instance `__dict__`
- **Non-data descriptor**: only `__get__` → the instance `__dict__` wins (functions, `cached_property`)

In [1]:
class Verbose:
    def __set_name__(self, owner, name):
        print(f"__set_name__: attached to {owner.__name__}.{name}")
        self.name = name
    def __get__(self, instance, owner):
        print(f"__get__(instance={instance!r}, owner={owner.__name__})")
        if instance is None:
            return self                     # accessed on the class itself
        return instance.__dict__.get(self.name, "<unset>")
    def __set__(self, instance, value):
        print(f"__set__({value!r})")
        instance.__dict__[self.name] = value

class Thing:
    x = Verbose()                           # __set_name__ runs right here

    def __repr__(self):
        return "Thing()"

t = Thing()
print(t.x)
t.x = 42
print(t.x)
print(Thing.x)

__set_name__: attached to Thing.x
__get__(instance=Thing(), owner=Thing)
<unset>
__set__(42)
__get__(instance=Thing(), owner=Thing)
42
__get__(instance=None, owner=Thing)


## 2. Reusable validated fields

Without descriptors, every validated attribute needs its own `@property` + setter. With one descriptor class you can reuse validation everywhere:

In [2]:
class Field:
    """Base descriptor: stores the value in the instance __dict__ under the same name."""
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(f"{self.name!r} is not set") from None
    def __set__(self, instance, value):
        instance.__dict__[self.name] = self.validate(value)
    def validate(self, value):
        return value

class Typed(Field):
    def __init__(self, kind):
        self.kind = kind
    def validate(self, value):
        if not isinstance(value, self.kind):
            raise TypeError(f"{self.name} must be {self.kind.__name__}, got {type(value).__name__}")
        return value

class Positive(Typed):
    def validate(self, value):
        value = super().validate(value)
        if value <= 0:
            raise ValueError(f"{self.name} must be > 0")
        return value

class OneOf(Field):
    def __init__(self, *options):
        self.options = options
    def validate(self, value):
        if value not in self.options:
            raise ValueError(f"{self.name} must be one of {self.options}")
        return value

class Product:
    name = Typed(str)
    price = Positive(float)
    quantity = Positive(int)
    category = OneOf("book", "toy", "food")

    def __init__(self, name, price, quantity, category):
        self.name, self.price, self.quantity, self.category = name, price, quantity, category

p = Product("Python Book", 499.0, 3, "book")
print(p.name, p.price, p.quantity, p.category, vars(p))
for bad in [("Pen", -5.0, 1, "book"), ("Pen", 5.0, 1, "car"), (42, 5.0, 1, "toy")]:
    try:
        Product(*bad)
    except (TypeError, ValueError) as e:
        print(type(e).__name__, "→", e)

Python Book 499.0 3 book {'name': 'Python Book', 'price': 499.0, 'quantity': 3, 'category': 'book'}
ValueError → price must be > 0
ValueError → category must be one of ('book', 'toy', 'food')
TypeError → name must be str, got int


This is exactly the pattern behind ORM/model fields like `models.CharField(max_length=50)` in Django or `Mapped[int] = mapped_column()` in SQLAlchemy.

### Where to store per-instance data?

❌ Never store it **on the descriptor** (`self.value = value`): there is only **one** descriptor object shared by all instances!

In [3]:
class BadField:
    def __get__(self, instance, owner):
        return self.value
    def __set__(self, instance, value):
        self.value = value                     # ❌ shared across ALL instances

class Account:
    balance = BadField()

a, b = Account(), Account()
a.balance = 100
b.balance = 5
print(a.balance)                               # 5 ?!

5


✅ Store it in `instance.__dict__` (as above), or in a `weakref.WeakKeyDictionary` keyed by instance when the class uses `__slots__`.

## 3. Re-implementing `property`

In [4]:
class MyProperty:
    def __init__(self, fget=None, fset=None):
        self.fget, self.fset = fget, fset
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return self.fget(instance)
    def __set__(self, instance, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(instance, value)
    def setter(self, fset):                  # enables @x.setter
        return type(self)(self.fget, fset)

class Circle:
    def __init__(self, r):
        self.r = r
    @MyProperty
    def area(self):
        return 3.14159 * self.r ** 2
    @MyProperty
    def radius(self):
        return self.r
    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("negative radius")
        self.r = value

c = Circle(2)
print(c.area)
c.radius = 3
print(c.radius, round(c.area, 2))
try:
    c.area = 1
except AttributeError as e:
    print("AttributeError:", e)

12.56636
3 28.27
AttributeError: can't set attribute


## 4. Re-implementing `classmethod`, `staticmethod` and `cached_property`

In [5]:
import types

class MyClassMethod:
    def __init__(self, func):
        self.func = func
    def __get__(self, instance, owner):
        return types.MethodType(self.func, owner)        # bind to the CLASS

class MyStaticMethod:
    def __init__(self, func):
        self.func = func
    def __get__(self, instance, owner):
        return self.func                                 # no binding at all

class MyCachedProperty:                                   # NON-data descriptor
    def __init__(self, func):
        self.func = func
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = self.func(instance)
        instance.__dict__[self.name] = value             # next lookups hit the instance dict first!
        return value

class Report:
    def __init__(self, data):
        self.data = data
    @MyClassMethod
    def from_csv_line(cls, line):
        return cls([int(x) for x in line.split(",")])
    @MyStaticMethod
    def describe():
        return "A report"
    @MyCachedProperty
    def total(self):
        print("  computing total...")
        return sum(self.data)

r = Report.from_csv_line("1,2,3,4")
print(r.data, Report.describe(), r.describe())
print(r.total)
print(r.total)            # cached: no "computing" message

[1, 2, 3, 4] A report A report
  computing total...
10
10


Because `MyCachedProperty` has no `__set__`, after the first access the **instance `__dict__` entry shadows it**: exactly how `functools.cached_property` works.

## 5. Functions *are* descriptors

In [6]:
def hello(self):
    return f"hello from {self!r}"

print(hasattr(hello, "__get__"), hasattr(hello, "__set__"))     # non-data descriptor
class A:
    pass
A.hello = hello                     # attach at runtime
print(A().hello())                  # bound automatically via function.__get__

True False
hello from <__main__.A object at 0x107bdb0e0>


## 6. `__slots__` use descriptors too

In [7]:
class Point:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

print(type(Point.__dict__["x"]))        # a member descriptor created for each slot
p = Point(1, 2)
print(Point.x.__get__(p, Point))

<class 'member_descriptor'>
1


## 7. When to use what

| Need | Use |
|---|---|
| One computed/validated attribute | `@property` |
| The same validation on many attributes/classes | a **descriptor** class |
| Expensive value computed once | `functools.cached_property` |
| Data model with validation & serialization | `dataclasses` + descriptors, or **Pydantic** (Module 15) |

## 📝 Summary
- Descriptors are class attributes with `__get__`/`__set__`/`__delete__`; `__set_name__` tells them their name.
- Data descriptors override instance attributes; non-data descriptors don't.
- `property`, methods, `classmethod`, `staticmethod`, `cached_property` and `__slots__` are all descriptors.
- Store per-instance state on the instance, not on the descriptor.

## 🏋️ Practice
1. Write a `Range(lo, hi)` descriptor and use it for `age = Range(0, 150)` in a `Person` class.
2. Write a `LoggedAttribute` descriptor that prints every change with the old and new value.
3. Write a `ReadOnly` descriptor that can be set **once** (in `__init__`) and then raises on reassignment.

<details><summary>Show solutions</summary>

```python
class Range:
    def __init__(self, lo, hi): self.lo, self.hi = lo, hi
    def __set_name__(self, owner, name): self.name = name
    def __get__(self, obj, owner): return self if obj is None else obj.__dict__[self.name]
    def __set__(self, obj, value):
        if not self.lo <= value <= self.hi:
            raise ValueError(f"{self.name} must be in [{self.lo}, {self.hi}]")
        obj.__dict__[self.name] = value

class LoggedAttribute:
    def __set_name__(self, owner, name): self.name = name
    def __get__(self, obj, owner): return self if obj is None else obj.__dict__.get(self.name)
    def __set__(self, obj, value):
        print(f"{self.name}: {obj.__dict__.get(self.name)!r} -> {value!r}")
        obj.__dict__[self.name] = value

class ReadOnly:
    def __set_name__(self, owner, name): self.name = name
    def __get__(self, obj, owner): return self if obj is None else obj.__dict__[self.name]
    def __set__(self, obj, value):
        if self.name in obj.__dict__:
            raise AttributeError(f"{self.name} is read-only")
        obj.__dict__[self.name] = value

class Person:
    age = Range(0, 150)
    name = ReadOnly()
    def __init__(self, name, age):
        self.name, self.age = name, age

p = Person("Asha", 30)
try:
    p.name = "X"
except AttributeError as e:
    print(e)
```
</details>